# 03 Source Diagnosis

## Business Question

哪个匿名注册来源贡献了2015-10-07后的主要新增变化？该变化是否可能受到数据口径影响？

## Analysis Objective

比较异常前后各`registered_via`的新增人数、占比和增量贡献，验证Source 4是否成为主要新增来源。本Notebook进行结构贡献分析，不解释具体业务原因。

## Data Used

- `data/processed/user_growth_profile.csv`
- `outputs/growth_daily_new_users.csv`
- `outputs/growth_daily_source_mix.csv`

## Key Metrics

- 各Source新增用户数、日均新增及占比。
- Source 4绝对增量和净增长贡献率。
- 非Source 4新增趋势。
- 10月7日至16日Source 4占比范围。


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
AS_OF_DATE = pd.Timestamp("2015-10-16")
ANALYSIS_START = pd.Timestamp("2015-07-01")
pd.set_option("display.max_columns", 50)


In [2]:
users = pd.read_csv(
    PROCESSED_DIR / "user_growth_profile.csv",
    usecols=["msno", "registration_date", "registered_via"],
    dtype={"msno": "string", "registered_via": "Int16"},
    parse_dates=["registration_date"],
)
assert users.msno.is_unique


In [3]:
periods = [
    ("Historical baseline", "2015-07-01", "2015-09-30"),
    ("Pre-break", "2015-10-01", "2015-10-06"),
    ("Break date", "2015-10-07", "2015-10-07"),
    ("Post-break", "2015-10-08", "2015-10-16"),
]
rows = []
for label, start, end in periods:
    frame = users[users.registration_date.between(start, end)]
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    for source, group in frame.groupby("registered_via"):
        rows.append({"period": label, "registered_via": int(source), "users": len(group),
                     "avg_daily_users": len(group) / days, "source_share": len(group) / len(frame)})
source_period = pd.DataFrame(rows)
display(source_period[source_period.registered_via.isin([3, 4, 7, 9])])


,period,registered_via,users,avg_daily_users,source_share
1,Historical baseline,3,187274,2035.586957,0.477294
2,Historical baseline,4,475,5.163043,0.001211
3,Historical baseline,7,56803,617.423913,0.144770
5,Historical baseline,9,146964,1597.434783,0.374558
10,Pre-break,3,9975,1662.500000,0.438057
11,Pre-break,4,134,22.333333,0.005885
12,Pre-break,7,3932,655.333333,0.172676
14,Pre-break,9,8687,1447.833333,0.381494
18,Break date,3,544,544.000000,0.079451
19,Break date,4,5189,5189.000000,0.757850


In [4]:
summary = []
for label, start, end in periods[1:]:
    frame = users[users.registration_date.between(start, end)]
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    s4 = int(frame.registered_via.eq(4).sum())
    summary.append({"period": label, "days": days, "total_users": len(frame),
                    "avg_daily_total": len(frame) / days, "source4_users": s4,
                    "avg_daily_source4": s4 / days, "source4_share": s4 / len(frame),
                    "avg_daily_non_source4": (len(frame) - s4) / days})
diagnosis = pd.DataFrame(summary)
pre = diagnosis.iloc[0]
diagnosis["total_daily_delta_vs_pre"] = diagnosis.avg_daily_total - pre.avg_daily_total
diagnosis["source4_daily_delta_vs_pre"] = diagnosis.avg_daily_source4 - pre.avg_daily_source4
diagnosis["source4_increment_contribution"] = (
    diagnosis.source4_daily_delta_vs_pre / diagnosis.total_daily_delta_vs_pre.replace(0, np.nan))
display(diagnosis)


,period,days,total_users,avg_daily_total,source4_users,avg_daily_source4,source4_share,avg_daily_non_source4,total_daily_delta_vs_pre,source4_daily_delta_vs_pre,source4_increment_contribution
0,Pre-break,6,22771,3795.166667,134,22.333333,0.005885,3772.833333,0.000000,0.000000,NaN
1,Break date,1,6847,6847.000000,5189,5189.000000,0.757850,1658.000000,3051.833333,5166.666667,1.692971
2,Post-break,9,82700,9188.888889,67046,7449.555556,0.810713,1739.333333,5393.722222,7427.222222,1.377012


In [5]:
daily = users.groupby("registration_date").size().rename("total_users").reset_index()
s4_daily = users[users.registered_via.eq(4)].groupby("registration_date").size().rename("source4_users").reset_index()
daily = daily.merge(s4_daily, on="registration_date", how="left").fillna({"source4_users": 0})
daily["non_source4_users"] = daily.total_users - daily.source4_users
daily["source4_share"] = daily.source4_users / daily.total_users
display(daily[daily.registration_date.between("2015-10-01", "2015-10-16")])


,registration_date,total_users,source4_users,non_source4_users,source4_share
92,2015-10-01,3177,20.0,3157.0,0.006295
93,2015-10-02,3604,26.0,3578.0,0.007214
94,2015-10-03,4659,24.0,4635.0,0.005151
95,2015-10-04,4728,14.0,4714.0,0.002961
96,2015-10-05,3235,20.0,3215.0,0.006182
97,2015-10-06,3368,30.0,3338.0,0.008907
98,2015-10-07,6847,5189.0,1658.0,0.757850
99,2015-10-08,7632,6189.0,1443.0,0.810928
100,2015-10-09,12413,9498.0,2915.0,0.765166
101,2015-10-10,11537,9056.0,2481.0,0.784953


## Data-definition Risk Check

在解释Source 4之前，业务侧需要核查：

- `registered_via`映射或编码规则是否在10月7日变化；
- 是否存在注册入口重分类；
- 是否存在批量用户迁移或补录；
- 是否存在埋点或产品版本变更；
- Source 4是否代表真实新增用户群。

公开数据不能回答这些问题。

## Source Diagnosis Conclusion

Source 4在异常前属于小规模来源，10月7日后成为主要新增来源并贡献主要新增增量。该结论是注册结构的描述性分解，不能将Source 4直接解释为广告渠道，也不证明具体业务原因。
